In [0]:
%sql
CREATE OR REPLACE TABLE data_warehouse_factory.gold.fct_final_operator_assignments AS
WITH 
-- 1. Połączenie planu z domyślnie przypisanym operatorem z tabeli assignments
assigning_default_operators AS (
    SELECT
        p.plan_date AS start_date,
        p.cell_code,
        p.cell_name,
        a.default_employee_key AS employee_key
    FROM data_warehouse_factory.silver.silver_production_plan p
    LEFT JOIN data_warehouse_factory.silver.silver_employee_assignments a 
        ON upper(trim(p.cell_code)) = upper(trim(a.cell_code)) 
        AND p.plan_date >= a.valid_from
),

-- 2. Wyciągnięcie zdarzeń typu Swap (rola zastępcza)
fct_swap_roles AS (
    SELECT
        production_date AS start_date,
        upper(trim(swap_cell)) AS cell_code,
        daily_event_start AS start_ts,
        daily_event_end AS end_ts,
        employee_key
    FROM data_warehouse_factory.silver.silver_events_daily
    WHERE event_type = 'Zmiana oprzyrządowania' OR event_type = '11-Swap role' OR event_type = 'Swap role'
),

-- 3. Wyciągnięcie nieobecności (Absence)
absences AS (
    SELECT 
        production_date AS start_date,
        employee_key AS absent_operator_key,
        daily_event_start AS start_ts,
        daily_event_end AS end_ts
    FROM data_warehouse_factory.silver.silver_events_daily
    WHERE event_type = 'Przerwa' OR event_type = 'Awaria' OR event_type = '62-Absence' OR event_type = 'Absence'
),

-- 4. Siatka komórek i domyślnych pracowników
grid_with_defaults AS (
    SELECT DISTINCT
        start_date,
        cell_code,
        employee_key
    FROM assigning_default_operators
),

-- 5. Zbiorczy zestaw wszystkich punktów granicznych czasów (Breakpoints)
all_time_points AS (
    -- Poczatek i koniec planu / zmiany
    SELECT start_date, cell_code, to_timestamp(concat(cast(start_date as string), ' 06:00:00')) AS bp_time FROM assigning_default_operators
    UNION DISTINCT
    SELECT start_date, cell_code, to_timestamp(concat(cast(start_date as string), ' 14:00:00')) AS bp_time FROM assigning_default_operators
    
    -- Punkty czasowe ze Swapów
    UNION DISTINCT
    SELECT start_date, cell_code, start_ts AS bp_time FROM fct_swap_roles
    UNION DISTINCT
    SELECT start_date, cell_code, end_ts AS bp_time FROM fct_swap_roles
    
    -- Punkty czasowe z Absencji (połączone z komórką, w której dany pracownik miał pracować)
    UNION DISTINCT
    SELECT g.start_date, g.cell_code, a.start_ts AS bp_time
    FROM grid_with_defaults g
    INNER JOIN absences a 
        ON g.start_date = a.start_date AND g.employee_key = a.absent_operator_key
    UNION DISTINCT
    SELECT g.start_date, g.cell_code, a.end_ts AS bp_time
    FROM grid_with_defaults g
    INNER JOIN absences a 
        ON g.start_date = a.start_date AND g.employee_key = a.absent_operator_key
),

-- 6. Budowanie ciągłych interwałów czasowych dla każdej komórki i dnia
creating_intervals AS (
    SELECT 
        start_date,
        cell_code,
        bp_time AS interval_start,
        LEAD(bp_time) OVER (PARTITION BY start_date, cell_code ORDER BY bp_time) AS interval_end
    FROM all_time_points
    WHERE bp_time IS NOT NULL
),

valid_intervals AS (
    SELECT * 
    FROM creating_intervals 
    WHERE interval_end IS NOT NULL 
      AND interval_start < interval_end
),

-- 7. Ocena priorytetów (1. Swap, 2. Absencja -> NULL, 3. Domyślny operator)
evaluated_intervals AS (
    SELECT 
        i.start_date,
        i.cell_code,
        i.interval_start,
        i.interval_end,
        
        CASE 
            WHEN s.employee_key IS NOT NULL THEN s.employee_key
            WHEN a.absent_operator_key IS NOT NULL THEN NULL
            ELSE g.employee_key
        END AS assigned_operator_key,

        ROW_NUMBER() OVER (
            PARTITION BY i.start_date, i.cell_code, i.interval_start 
            ORDER BY 
                CASE 
                    WHEN s.employee_key IS NOT NULL THEN 1  -- Swap
                    WHEN a.absent_operator_key IS NOT NULL THEN 2 -- Absencja
                    ELSE 3 -- Domyślny
                END ASC
        ) AS rn
    FROM valid_intervals i
    LEFT JOIN grid_with_defaults g 
        ON i.start_date = g.start_date 
        AND i.cell_code = g.cell_code
    LEFT JOIN fct_swap_roles s 
        ON i.start_date = s.start_date 
        AND i.cell_code = s.cell_code 
        AND i.interval_start >= s.start_ts 
        AND i.interval_end <= s.end_ts
    LEFT JOIN absences a 
        ON i.start_date = a.start_date 
        AND g.employee_key = a.absent_operator_key
        AND i.interval_start >= a.start_ts 
        AND i.interval_end <= a.end_ts
)

SELECT 
    start_date,
    cell_code,
    date_format(interval_start, 'HH:mm:ss') AS start_time,
    date_format(interval_end, 'HH:mm:ss') AS end_time,
    interval_start,
    interval_end,
    assigned_operator_key
FROM evaluated_intervals
WHERE rn = 1 AND assigned_operator_key IS NOT NULL;